In [ ]:
# Librerie di sistema e utilità
import os
import warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# Librerie per elaborazione audio
import librosa

# PyTorch
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torchvision import transforms
import timm

# Ignoriamo i warning
warnings.filterwarnings("ignore")

print("Librerie importate con successo!")
print(f"PyTorch versione: {torch.__version__}")
print(f"timm versione: {timm.__version__}")

In [ ]:
class Config:
    def __init__(self):
        # Imposta i percorsi di base in base all'ambiente
        self.COMPETITION_NAME = "birdclef-2025"
        self.BASE_DIR = f"/kaggle/input/{self.COMPETITION_NAME}"
        self.OUTPUT_DIR = "/kaggle/working"
        self.MODELS_DIR = "/kaggle/input"  # Per i modelli pre-addestrati
            
        # Imposta subito i percorsi derivati per l'ambiente Kaggle
        self._setup_derived_paths()
        
        # Parametri per il preprocessing audio
        self.SR = 32000      # Sample rate
        self.DURATION = 5    # Durata dei clip in secondi
        self.N_MELS = 224    # Numero di bande Mel
        self.N_FFT = 2048    # Dimensione finestra FFT
        self.HOP_LENGTH = 512  # Hop length per STFT
        self.FMIN = 48       # Frequenza minima per lo spettrogramma Mel
        self.FMAX = 16000    # Frequenza massima
        self.POWER = 2       # Esponente per calcolo spettrogramma
            
        # Parametri per il device
        self.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Parametri per inference/submission
        self.TEST_CLIP_DURATION = 5  # Durata dei segmenti per la predizione (secondi)
        self.N_CLASSES = 0  # Sarà impostato dopo aver caricato i dati

    def _setup_derived_paths(self):
        """Imposta i percorsi derivati basati su BASE_DIR"""
        self.TRAIN_AUDIO_DIR = os.path.join(self.BASE_DIR, "train_audio")
        self.TEST_SOUNDSCAPES_DIR = os.path.join(self.BASE_DIR, "test_soundscapes")
        self.TRAIN_CSV_PATH = os.path.join(self.BASE_DIR, "train.csv")
        self.TAXONOMY_CSV_PATH = os.path.join(self.BASE_DIR, "taxonomy.csv") 
        self.SAMPLE_SUB_PATH = os.path.join(self.BASE_DIR, "sample_submission.csv")

# Inizializza la configurazione
config = Config()

print(f"Device utilizzato: {config.DEVICE}")
print(f"Directory Test Soundscapes: {config.TEST_SOUNDSCAPES_DIR}")
print(f"Path Sample Submission: {config.SAMPLE_SUB_PATH}")

In [ ]:
def load_metadata():
    """
    Carica e prepara i metadati dal file CSV di training.
    
    Returns:
        tuple: all_species
    """
    print(f"Caricamento metadati da: {config.TRAIN_CSV_PATH}")
    train_df = pd.read_csv(config.TRAIN_CSV_PATH)
    sample_sub_df = pd.read_csv(config.SAMPLE_SUB_PATH)
    
    # Estrai tutte le etichette uniche
    train_primary_labels = train_df['primary_label'].unique()
    train_secondary_labels = set([lbl for sublist in train_df['secondary_labels'].apply(eval) 
                                 for lbl in sublist if lbl])
    submission_species = sample_sub_df.columns[1:].tolist()  # Escludi row_id
    
    # Combina tutte le possibili etichette
    all_species = sorted(list(set(train_primary_labels) | train_secondary_labels | set(submission_species)))
    N_CLASSES = len(all_species)
    config.N_CLASSES = N_CLASSES  # Aggiorna il numero di classi nella configurazione
    
    print(f"Numero totale di specie trovate: {N_CLASSES}")
    
    return all_species

# Carica le specie
all_species = load_metadata()

In [ ]:
# Crea una singola istanza della trasformazione MelSpectrogram da riutilizzare
mel_transform = T.MelSpectrogram(
    sample_rate=config.SR,
    n_fft=config.N_FFT,
    win_length=None,
    hop_length=config.HOP_LENGTH,
    f_min=config.FMIN,
    f_max=config.FMAX,
    n_mels=config.N_MELS,
    window_fn=torch.hann_window,
    power=config.POWER,
    normalized=False,
    onesided=True,
    norm="slaney",
    mel_scale="slaney"
)

# Funzione di conversione a dB e normalizzazione
def amplitude_to_db(spectrogram):
    """Converti spettrogramma in scala dB e normalizza tra 0-1"""
    # Converti in dB
    spectrogram_db = 10.0 * torch.log10(torch.clamp(spectrogram, min=1e-10))
    
    # Normalizza
    min_val = torch.min(spectrogram_db)
    max_val = torch.max(spectrogram_db)
    if max_val > min_val:
        return (spectrogram_db - min_val) / (max_val - min_val)
    else:
        return torch.zeros_like(spectrogram_db)

In [ ]:
class EfficientNetBirdClassifier(nn.Module):
    def __init__(self, num_classes=config.N_CLASSES, pretrained=False, model_name='efficientnet_b0'):
        super(EfficientNetBirdClassifier, self).__init__()
        
        # Crea il modello
        self.efficientnet = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0  # Rimuovi il classificatore originale
        )
        
        # Ottieni la dimensione dell'output del feature extractor
        if hasattr(self.efficientnet, 'num_features'):
            classifier_in_features = self.efficientnet.num_features
        elif hasattr(self.efficientnet, 'classifier'):
            classifier_in_features = self.efficientnet.classifier.in_features
        else:
            # Valore predefinito per EfficientNet-B0
            classifier_in_features = 1280
        
        # Sostituisci il classificatore semplice con una MLP con dropout
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),  # Primo dropout significativo
            nn.Linear(classifier_in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),  # Secondo dropout più leggero
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        # Se l'input è un'immagine a 1 canale, replicala su 3 canali
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        
        # Passa l'input attraverso il backbone per ottenere le features
        features = self.efficientnet(x)
        
        # Passa le feature attraverso il classificatore
        output = self.classifier(features)
        
        return output

In [ ]:
# Percorso del modello pre-addestrato (modifica secondo necessità)
model_path = "/kaggle/input/mio-modello-addestrato/birdclef_efficientNET_dataAugPaper_timm_best.pth"

# Inizializza il modello
model = EfficientNetBirdClassifier(num_classes=config.N_CLASSES, pretrained=False).to(config.DEVICE)

# Carica il modello pre-addestrato
try:
    print(f"Caricamento del modello pre-addestrato da {model_path}...")
    checkpoint = torch.load(model_path, map_location=config.DEVICE)
    
    # Verifica se è un dict con model_state_dict o direttamente state_dict
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Modello caricato con successo (epoca: {checkpoint.get('epoch', 'N/A')})")
    else:
        model.load_state_dict(checkpoint)
        print("Modello caricato con successo")
    
    model.eval()  # Imposta il modello in modalità valutazione
except Exception as e:
    print(f"Errore nel caricamento del modello: {e}")
    raise

In [ ]:
train_soundscapes_dir = "/kaggle/input/birdclef-2025/train_soundscapes"
output_dir = "/kaggle/working/pseudo_labeled_data"

In [ ]:
def assign_labels_to_soundscapes(model, soundscapes_dir, device=config.DEVICE, 
                               num_segments=5, confidence_threshold=0.7, 
                               consistency_threshold=0.6):
    """
    Assegna etichette ai soundscapes usando un modello pre-addestrato con controlli
    di confidenza e consistenza.
    
    Args:
        model: Modello PyTorch addestrato
        soundscapes_dir: Directory contenente i file audio da etichettare
        device: Device per inferenza ('cuda' o 'cpu')
        num_segments: Numero di segmenti casuali da estrarre da ciascun file
        confidence_threshold: Soglia minima di confidenza per accettare una predizione
        consistency_threshold: Frazione di segmenti che devono concordare sulla classe
        
    Returns:
        pd.DataFrame: DataFrame con le etichette assegnate e relative metriche
    """
    model.to(device)
    model.eval()
    
    labeled_data = []
    skipped_files = []
    
    # Set seed per riproducibilità
    np.random.seed(42)
    
    # Lista di tutti i file audio
    audio_files = [f for f in os.listdir(soundscapes_dir) if f.endswith('.ogg')]
    print(f"Trovati {len(audio_files)} file audio da etichettare")
    
    for audio_file in tqdm(audio_files, desc="Elaborazione soundscapes"):
        # Carica il file audio
        audio_path = os.path.join(soundscapes_dir, audio_file)
        try:
            signal, sr = librosa.load(audio_path, sr=config.SR)
        except Exception as e:
            print(f"Errore caricamento file {audio_file}: {e}")
            skipped_files.append({"filename": audio_file, "reason": "error_loading"})
            continue
        
        # Estrai n segmenti casuali
        segments = []
        segment_length = config.SR * config.DURATION
        
        if len(signal) > segment_length * 1.5:  # File abbastanza lungo
            for _ in range(num_segments):
                max_start = len(signal) - segment_length
                if max_start <= 0:
                    # File troppo corto, usa tutto il file
                    start_idx = 0
                else:
                    start_idx = np.random.randint(0, max_start)
                segments.append(signal[start_idx:start_idx + segment_length])
        else:
            # Audio troppo corto, usa tutto e padda se necessario
            if len(signal) < segment_length:
                signal = np.pad(signal, (0, segment_length - len(signal)), mode='constant')
            segments.append(signal)
        
        
        # Delta shifts per TTA (in campioni)
        delta_shifts = [-1600, -800, 0, 800, 1600]  # ±25ms, ±50ms con SR=32000
        
        all_predictions = []
        
        for segment in segments:
            # Per ogni segmento, applica i delta shift
            segment_predictions = []
            
            for shift in delta_shifts:
                # Applica lo shift
                if shift != 0:
                    shifted_segment = np.roll(segment, shift)
                    # Azzera i bordi per evitare artefatti
                    if shift > 0:
                        shifted_segment[:shift] = 0
                    else:
                        shifted_segment[shift:] = 0
                else:
                    shifted_segment = segment
                    
                # Calcola spettrogramma Mel usando il segmento shiftato
                mel_spec = librosa.feature.melspectrogram(
                    y=shifted_segment,  # <-- Usa il segmento shiftato
                    sr=config.SR,
                    n_fft=config.N_FFT,
                    hop_length=config.HOP_LENGTH,
                    n_mels=config.N_MELS,
                    fmin=config.FMIN,
                    fmax=config.FMAX
                )
            
            # Converti in scala logaritmica (dB) e normalizza
            log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
            min_val = np.min(log_mel_spec)
            max_val = np.max(log_mel_spec)
            if max_val > min_val:
                log_mel_spec = (log_mel_spec - min_val) / (max_val - min_val)
            else:
                log_mel_spec = np.zeros_like(log_mel_spec)
            
            # Prepara il tensor per il modello
            log_mel_spec = np.expand_dims(np.expand_dims(log_mel_spec, axis=0), axis=0)
            input_tensor = torch.tensor(log_mel_spec, dtype=torch.float32).to(device)
            
            # Resize a 224x224 come nel training
            resize_transform = transforms.Resize((224, 224), 
                interpolation=transforms.InterpolationMode.BICUBIC)
            input_tensor = resize_transform(input_tensor)

            # Effettua predizione
            with torch.no_grad():
                output = model(input_tensor)
                scores = torch.sigmoid(output).cpu().numpy()[0]
            
            segment_predictions.append(scores)
        
        # Media delle predizioni di tutti gli shift per questo segmento
        avg_predictions = np.mean(segment_predictions, axis=0)
        all_predictions.append(avg_predictions)
        
        # Dopo aver elaborato tutti i segmenti, calcola una media generale di tutte le predizioni
        file_avg_prediction = np.mean(all_predictions, axis=0)
        
        # Trova la classe con lo score più alto nella media generale del file
        max_class_idx = np.argmax(file_avg_prediction)
        max_confidence = file_avg_prediction[max_class_idx]
        predicted_class = all_species[max_class_idx]
        
        # Verifica consistenza tra i segmenti
        # Conta quanti segmenti concordano sulla classe principale
        top_classes = [np.argmax(p) for p in all_predictions]
        num_agreeing = sum(1 for c in top_classes if c == max_class_idx)
        consistency_score = num_agreeing / len(all_predictions)
        
        # Calcola metriche aggiuntive per il rapporto
        avg_confidence = np.mean([p[max_class_idx] for p in all_predictions])
        std_confidence = np.std([p[max_class_idx] for p in all_predictions])
        
        # Assegna etichetta solo se supera la soglia di confidenza e consistenza
        if max_confidence > confidence_threshold and consistency_score >= consistency_threshold:
            labeled_data.append({
                'filename': audio_file,
                'predicted_label': predicted_class,
                'confidence': max_confidence,
                'consistency': consistency_score,
                'avg_confidence': avg_confidence,
                'std_confidence': std_confidence,
                'num_segments': len(all_predictions)
            })
        else:
            skipped_files.append({
                'filename': audio_file,
                'max_confidence': max_confidence,
                'consistency': consistency_score,
                'reason': 'low_confidence' if max_confidence <= confidence_threshold else 'inconsistent'
            })
    
    # Crea DataFrames
    labeled_df = pd.DataFrame(labeled_data)
    skipped_df = pd.DataFrame(skipped_files)
    
    # Analisi dettagliata dei motivi di scarto
    low_confidence_files = skipped_df[skipped_df['reason'] == 'low_confidence']
    inconsistent_files = skipped_df[skipped_df['reason'] == 'inconsistent']
    error_files = skipped_df[skipped_df['reason'] == 'error_loading']
    
    print(f"\n{'='*50}")
    print(f"REPORT COMPLETO DI ETICHETTATURA:")
    print(f"{'='*50}")
    print(f"- File totali trovati:       {len(audio_files)}")
    print(f"- File etichettati:          {len(labeled_df)} ({len(labeled_df)/len(audio_files):.1%})")
    print(f"- File saltati totali:       {len(skipped_df)} ({len(skipped_df)/len(audio_files):.1%})")
    print(f"  ├─ Bassa confidenza:       {len(low_confidence_files)} ({len(low_confidence_files)/len(audio_files):.1%})")
    print(f"  ├─ Bassa consistenza:      {len(inconsistent_files)} ({len(inconsistent_files)/len(audio_files):.1%})")
    print(f"  └─ Errori di caricamento:  {len(error_files)} ({len(error_files)/len(audio_files):.1%})")
    
    if len(labeled_df) > 0:
        print(f"\nSTATISTICHE FILE ETICHETTATI:")
        print(f"- Confidenza media:         {labeled_df['confidence'].mean():.4f} (min: {labeled_df['confidence'].min():.2f}, max: {labeled_df['confidence'].max():.2f})")
        print(f"- Consistenza media:        {labeled_df['consistency'].mean():.4f} (min: {labeled_df['consistency'].min():.2f}, max: {labeled_df['consistency'].max():.2f})")
        
        # Distribuzioni di confidenza
        print("\nDISTRIBUZIONE CONFIDENZA:")
        bins = [0.7, 0.8, 0.9, 0.95, 1.0]
        hist, _ = np.histogram(labeled_df['confidence'], bins=bins)
        for i in range(len(bins)-1):
            print(f"  {bins[i]:.2f}-{bins[i+1]:.2f}: {hist[i]} file ({hist[i]/len(labeled_df):.1%})")
        
        # Distribuzioni di classi (top 10)
        print(f"\nDISTRIBUZIONE CLASSI (TOP 10):")
        class_counts = labeled_df['predicted_label'].value_counts()
        for label, count in class_counts.head(10).items():
            print(f"  {label}: {count} file ({count/len(labeled_df):.1%})")
    
    print(f"{'='*50}")
    
    return labeled_df, skipped_df

In [ ]:
def prepare_pseudo_labeled_data(labeled_df, train_soundscapes_dir, output_dir):
    """
    Prepara i dati pseudo-etichettati in un formato compatibile con train.csv
    
    Args:
        labeled_df: DataFrame con le etichette assegnate da assign_labels_to_soundscapes
        train_soundscapes_dir: Directory contenente i file audio originali
        output_dir: Directory dove salvare i file preparati e il CSV
    """
    import os
    import pandas as pd
    import shutil
    from tqdm.notebook import tqdm
    
    # Crea directory di output se non esiste
    os.makedirs(output_dir, exist_ok=True)
    
    # Crea una sottodirectory per i file pseudo-etichettati
    pseudo_audio_dir = os.path.join(output_dir, "pseudo_audio")
    os.makedirs(pseudo_audio_dir, exist_ok=True)
    
    # Per ogni specie, crea una directory specifica
    for species in labeled_df['predicted_label'].unique():
        os.makedirs(os.path.join(pseudo_audio_dir, species), exist_ok=True)
    
    # Lista per memorizzare le righe del CSV di train
    pseudo_train_rows = []
    
    # Processa ogni file etichettato
    for idx, row in tqdm(labeled_df.iterrows(), desc="Preparazione dati", total=len(labeled_df)):
        # Percorso di origine e destinazione
        src_path = os.path.join(train_soundscapes_dir, row['filename'])
        
        # Crea un nuovo nome file con prefisso 'pseudo_'
        new_filename = f"pseudo_{row['filename']}"
        
        # Percorso di destinazione nel formato species/filename
        species_dir = os.path.join(pseudo_audio_dir, row['predicted_label'])
        dst_path = os.path.join(species_dir, new_filename)
        
        # Copia il file audio
        shutil.copy2(src_path, dst_path)
        
        # Crea una riga per il CSV nel formato di train.csv
        new_row = {
            'primary_label': row['predicted_label'],
            'secondary_labels': [''],  # Lista vuota come stringa
            'type': [''],              # Lista vuota come stringa
            'filename': f"{row['predicted_label']}/{new_filename}",
            'collection': 'Pseudo',
            'rating': 0,
            'url': '',
            'latitude': '',
            'longitude': '',
            'scientific_name': row['predicted_label'],  # Usiamo lo stesso valore dell'etichetta
            'common_name': row['predicted_label'],      # Usiamo lo stesso valore dell'etichetta
            'author': 'Pseudo-labeled',
            'license': 'N/A',
            # Aggiungi colonne extra per tracciare la qualità dell'etichettatura
            'confidence': row['confidence'],
            'consistency': row['consistency']
        }
        
        pseudo_train_rows.append(new_row)
    
    # Crea DataFrame e salva come CSV
    pseudo_train_df = pd.DataFrame(pseudo_train_rows)
    csv_path = os.path.join(output_dir, "pseudo_train.csv")
    pseudo_train_df.to_csv(csv_path, index=False)
    
    print(f"\nDati pseudo-etichettati preparati in {output_dir}")
    print(f"- File CSV: {csv_path}")
    print(f"- File audio: {pseudo_audio_dir}")
    print(f"- Totale esempi: {len(pseudo_train_df)}")
    
    return pseudo_train_df

In [ ]:
# 2. Esegui la pseudo-etichettatura
labeled_df, skipped_df = assign_labels_to_soundscapes(
    model=model,
    soundscapes_dir=train_soundscapes_dir,
    confidence_threshold=0.7,
    consistency_threshold=0.6
)

# 3. Salva i risultati intermedi per analisi
labeled_df.to_csv("/kaggle/working/labeled_soundscapes.csv", index=False)
skipped_df.to_csv("/kaggle/working/skipped_soundscapes.csv", index=False)

# 4. Prepara i dati in formato train.csv e salva i file
pseudo_train_df = prepare_pseudo_labeled_data(
    labeled_df=labeled_df,
    train_soundscapes_dir=train_soundscapes_dir,
    output_dir=output_dir
)